# 4.2 Frequent itemset mining (association rules)

**A deliberate exception to "one table throughout":** association rules
need basket-shaped (transaction) data, not the customer-attribute table
used everywhere else in this folder. Rather than invent a new toy dataset,
this notebook reuses the *exact* tiny worked examples already in the
slides -- the "Golden iPhone / Purple case" basket-of-4 example, and the
"iPhone X / Samsung / LG TV" exercise -- so it's still the same course
example, just the one the slides themselves already built for this
technique.

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("common"))
sys.path.insert(0, os.path.abspath("../common"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

CUSTOMERS_PATH = "data/customers.csv" if os.path.exists("data/customers.csv") else "../data/customers.csv"
df = pd.read_csv(CUSTOMERS_PATH)
df.head()

,customer_id,age,region,tenure_months,income_k,monthly_usage,support_calls,discount_pct,satisfaction,monthly_spend,churned
0,C0001,45.7,South,90.7,78.1,8.3,2,0,8.12,39.35,0
1,C0002,29.5,North,54.9,58.4,27.3,6,5,5.80,51.69,0
2,C0003,51.0,West,28.6,73.3,13.9,5,0,5.48,37.75,1
3,C0004,53.3,West,22.9,106.5,13.3,6,10,9.25,34.31,1
4,C0005,18.6,West,45.7,35.5,67.2,6,15,7.26,52.28,0


## The worked example: 4 baskets

In [2]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

baskets = [
    ["Golden iPhone", "Purple case", "MacBook Pro"],
    ["Golden iPhone", "Purple case", "Bar of chocolate"],
    ["Golden iPhone", "Purple case", "Golden iPhone"],  # basket 3 in the slide repeats "Golden iPhone" as its own row
    ["Purple case", "Six pack"],
]
# basket 3 in the slide is really {Golden iPhone, Purple case} -- de-duplicate
baskets[2] = ["Golden iPhone", "Purple case"]

te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)
basket_df

,Bar of chocolate,Golden iPhone,MacBook Pro,Purple case,Six pack
0,False,True,True,True,False
1,True,True,False,True,False
2,False,True,False,True,False
3,False,False,False,True,True


## Support, confidence, lift -- by hand, then with mlxtend

In [3]:
n = len(baskets)
support_golden = basket_df["Golden iPhone"].sum() / n
support_purple = basket_df["Purple case"].sum() / n
support_both = (basket_df["Golden iPhone"] & basket_df["Purple case"]).sum() / n
confidence = support_both / support_golden
lift = confidence / support_purple

print(f"support(Golden iPhone)              = {support_golden:.2f}")
print(f"support(Purple case)                = {support_purple:.2f}")
print(f"support(Golden iPhone & Purple case) = {support_both:.2f}")
print(f"confidence(Golden iPhone -> Purple case) = {confidence:.2f}")
print(f"lift(Golden iPhone -> Purple case)       = {lift:.2f}   (>1 means positively associated)")

support(Golden iPhone)              = 0.75
support(Purple case)                = 1.00
support(Golden iPhone & Purple case) = 0.75
confidence(Golden iPhone -> Purple case) = 1.00
lift(Golden iPhone -> Purple case)       = 1.00   (>1 means positively associated)


In [4]:
frequent_itemsets = apriori(basket_df, min_support=0.25, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
rules[["antecedents", "consequents", "support", "confidence", "lift"]].sort_values("lift", ascending=False)

/usr/local/lib/python3.11/dist-packages/mlxtend/frequent_patterns/association_rules.py:184: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


,antecedents,consequents,support,confidence,lift
0,frozenset({Bar of chocolate}),frozenset({Golden iPhone}),0.25,1.00,1.333333
2,frozenset({MacBook Pro}),frozenset({Golden iPhone}),0.25,1.00,1.333333
11,"frozenset({MacBook Pro, Purple case})",frozenset({Golden iPhone}),0.25,1.00,1.333333
12,frozenset({MacBook Pro}),"frozenset({Golden iPhone, Purple case})",0.25,1.00,1.333333
8,"frozenset({Purple case, Bar of chocolate})",frozenset({Golden iPhone}),0.25,1.00,1.333333
9,frozenset({Bar of chocolate}),"frozenset({Golden iPhone, Purple case})",0.25,1.00,1.333333
4,frozenset({Purple case}),frozenset({Golden iPhone}),0.75,0.75,1.000000
1,frozenset({Bar of chocolate}),frozenset({Purple case}),0.25,1.00,1.000000
3,frozenset({Golden iPhone}),frozenset({Purple case}),0.75,1.00,1.000000
7,"frozenset({Golden iPhone, Bar of chocolate})",frozenset({Purple case}),0.25,1.00,1.000000


Same three numbers as the by-hand calculation above, just computed for
*every* itemset at once -- this is what the Apriori algorithm buys you once
there are too many possible item combinations to check by hand.

## The exercise basket: a second worked pass, and the Apriori algorithm

In [5]:
exercise_baskets = [
    ["iPhone X", "Samsung S8", "LG 55\" TV", "Samsung S9"],
    ["iPhone X", "Samsung S9", "LG 55\" TV"],
    ["LG 55\" TV", "Samsung S9", "iPhone X"],
    ["Samsung S8", "Samsung S9"],
]

te2 = TransactionEncoder()
ex_df = pd.DataFrame(te2.fit(exercise_baskets).transform(exercise_baskets), columns=te2.columns_)

print("Apriori, step by step (support of single items -> pairs -> triples):")
for k in (1, 2, 3):
    itemsets_k = apriori(ex_df, min_support=0.01, use_colnames=True)
    itemsets_k = itemsets_k[itemsets_k["itemsets"].apply(len) == k]
    print(f"\n-- size-{k} itemsets --")
    print(itemsets_k.sort_values("support", ascending=False).to_string(index=False))

Apriori, step by step (support of single items -> pairs -> triples):

-- size-1 itemsets --
 support                itemsets
    1.00 frozenset({Samsung S9})
    0.75  frozenset({LG 55" TV})
    0.75   frozenset({iPhone X})
    0.50 frozenset({Samsung S8})

-- size-2 itemsets --
 support                            itemsets
    0.75  frozenset({LG 55" TV, Samsung S9})
    0.75    frozenset({LG 55" TV, iPhone X})
    0.75   frozenset({iPhone X, Samsung S9})
    0.50 frozenset({Samsung S8, Samsung S9})
    0.25  frozenset({LG 55" TV, Samsung S8})
    0.25   frozenset({Samsung S8, iPhone X})

-- size-3 itemsets --
 support                                       itemsets
    0.75   frozenset({LG 55" TV, iPhone X, Samsung S9})
    0.25 frozenset({LG 55" TV, Samsung S8, Samsung S9})
    0.25   frozenset({LG 55" TV, Samsung S8, iPhone X})
    0.25  frozenset({Samsung S8, iPhone X, Samsung S9})


This is exactly the Apriori exercise from the slides worked in code: start
from single items, keep only itemsets meeting the minimum support, then
only *extend* itemsets that survived the previous round into the next size
-- the "anti-monotone" pruning rule that makes Apriori tractable on baskets
too large to enumerate by brute force.

**Takeaway**: support/confidence/lift are simple counting exercises on a
handful of baskets like these -- Apriori's real contribution is *scaling*
that counting up to thousands of items/transactions without checking every
possible combination.